In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from keras.src.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.src.models import Sequential
from keras.src.callbacks import EarlyStopping
from keras.src.applications.resnet import ResNet50
from keras.src.applications.mobilenet_v3 import MobileNetV3Small

In [2]:
class MultimodalClassifier:
    def __init__(self):
        self.text_model = None
        self.image_model = None
        self.url_model = None
        self.image_dataset = None

    def train(self, text_data, text_labels, image_data, image_labels, url_data, url_labels):
        print("Begin training text")
        self.text_model = VotingClassifier([
            ('rf', RandomForestClassifier(random_state=42, n_jobs=4)),
            ('sgd', SGDClassifier(n_jobs=4)),
            ('svc', LinearSVC())
        ])

        X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
            text_data, text_labels, test_size=0.2, random_state=42
        )

        self.text_model.fit(X_text_train, y_text_train)
        text_pred = self.text_model.predict(X_text_test)
        text_accuracy = accuracy_score(y_text_test, text_pred)
        print(f"Text model accuracy: {text_accuracy:.4f}")

        print("Begin training image")
        self.build_image_model()

        X_image_train, X_image_test, y_image_train, y_image_test = train_test_split(
            image_data, image_labels, test_size=0.2, random_state=42
        )

        history = self.image_model.fit(
            X_image_train, y_image_train,
            validation_data=(X_image_test, y_image_test),
            epochs=10, batch_size=32, verbose=1,
            callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
        )

        img_pred = self.image_model.predict(X_image_test)
        img_pred_classes = (img_pred > 0.5).astype(int)
        img_accuracy = accuracy_score(y_image_test, img_pred_classes)
        print(f"Image model accuracy: {img_accuracy:.4f}")

        print("Begin training URL")
        self.url_model = VotingClassifier([
            ('rf', RandomForestClassifier(random_state=42, n_jobs=4)),
            ('sgd', SGDClassifier(n_jobs=4)),
            ('svc', LinearSVC())
        ])

        X_url_train, X_url_test, y_url_train, y_url_test = train_test_split(
            url_data, url_labels, test_size=0.2, random_state=42
        )

        self.url_model.fit(X_url_train, y_url_train)
        url_pred = self.url_model.predict(X_url_test)
        url_accuracy = accuracy_score(y_url_test, url_pred)
        print(f"URL model accuracy: {url_accuracy:.4f}")

    def predict(self, text_data=None, image_data=None, url_data=None):
        predictions = []

        if text_data is not None and self.text_model is not None:
            text_pred = self.text_model.predict(text_data)
            predictions.append(text_pred)

        if image_data is not None and self.image_model is not None:
            img_pred = self.image_model.predict(image_data)
            img_pred_classes = (img_pred > 0.5).astype(int)
            predictions.append(img_pred_classes)

        if url_data is not None and self.url_model is not None:
            url_pred = self.url_model.predict(url_data)
            predictions.append(url_pred)

        

        stacked_preds = np.stack(predictions, axis=0)

        return (np.mean(stacked_preds, axis=0) > 0.5).astype(int)

    def build_image_model(self):
        base_model = MobileNetV3Small(
            include_top=False,
            input_shape=(224, 224, 3)
        )
        
        self.image_model = Sequential([
            base_model,
            GlobalAveragePooling2D(),
            Dropout(0.5),
            Dense(128, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])

        self.image_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [3]:
import joblib

text_lemmatized_tfidf = joblib.load('./text_data/enron_lemmatized_tfidf.pkl')
text_X = text_lemmatized_tfidf['features']
text_y = text_lemmatized_tfidf['labels']

In [4]:
image_X = np.load('./image_data/data_X.npy')
image_y = np.load('./image_data/data_y.npy')

In [5]:
import pandas as pd
df = pd.read_csv("./url_data/malicious_phish_preprocessed_100k.csv")
url_X = df.drop(columns=["Unnamed: 0", "Unnamed: 0.1", "url", "type"])
url_y = df["type"]
del df

In [6]:
url_X = pd.get_dummies(url_X, columns=["tld"], drop_first=True)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
url_y = le.fit_transform(url_y)

url_y = (url_y == 0).astype(int)

In [7]:
print(text_X)
print(text_y)

  (0, 20702)	0.4080648750394073
  (0, 22980)	0.38639074346714675
  (0, 12386)	0.4834906642505629
  (0, 30309)	0.4824135805761483
  (0, 5992)	0.46658243213844675
  (1, 15494)	0.42352863530105617
  (1, 13425)	0.0594387659857789
  (1, 20013)	0.04039288931043278
  (1, 20471)	0.0594387659857789
  (1, 10831)	0.029986675399526246
  (1, 15142)	0.02777470884865176
  (1, 29656)	0.01893533431827908
  (1, 19051)	0.021344396562375727
  (1, 31410)	0.03021868710701771
  (1, 13523)	0.02456891712063736
  (1, 28747)	0.0370567875723566
  (1, 580)	0.037944433496692714
  (1, 7365)	0.022678641629967904
  (1, 27937)	0.032748977812648745
  (1, 26826)	0.020702645228897365
  (1, 32399)	0.01827944222935332
  (1, 12454)	0.052047998079904845
  (1, 31640)	0.07243337452738216
  (1, 24983)	0.032996163322207804
  (1, 30523)	0.03477614499217725
  :	:
  (33713, 24827)	0.33190504733915727
  (33713, 18717)	0.3674813634394411
  (33713, 20916)	0.3288702546006061
  (33713, 18162)	0.3747826284994613
  (33713, 18039)	0.2649260

In [8]:
print(url_X, url_y)

       url_length  hostname_length  path_length  num_dots  num_slashes  \
0              43               20           16         2            4   
1              52                9           36         3            7   
2              47               24           16         4            4   
3              73               31           35         3            5   
4              30               16            7         2            3   
...           ...              ...          ...       ...          ...   
99995          40               18           15         2            3   
99996          42               14           21         2            4   
99997          22               15            0         1            2   
99998          52                7           38         1            5   
99999         104               22           10         2            3   

       num_digits  num_special_chars  has_https  has_ip  has_port  ...  \
0               2                  7 

In [9]:
mmc = MultimodalClassifier()

In [10]:
mmc.train(text_X, text_y, image_X, image_y, url_X, url_y)

Begin training text
Text model accuracy: 0.9889
Begin training image
Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 48s 578ms/step - accuracy: 0.9357 - loss: 0.1774 - val_accuracy: 0.8333 - val_loss: 1.5598
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 23s 530ms/step - accuracy: 0.9972 - loss: 0.0117 - val_accuracy: 0.7902 - val_loss: 2.7502
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 23s 531ms/step - accuracy: 0.9996 - loss: 0.0049 - val_accuracy: 0.7845 - val_loss: 3.2320
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 25s 557ms/step - accuracy: 0.9986 - loss: 0.0084 - val_accuracy: 0.7557 - val_loss: 6.8208
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 204ms/step
Image model accuracy: 0.8333
Begin training URL


E:\Python Tests\AI\.venv\lib\site-packages\sklearn\svm\_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


URL model accuracy: 0.8441


In [11]:
text_X_test, _, text_y_test, _ = train_test_split(text_X, text_y, random_state=42, train_size=2000)

In [12]:
url_X_test, _, url_y_test, _ = train_test_split(url_X, url_y, random_state=42, train_size=2000)

url_y_test

In [13]:
preds = mmc.predict(text_data=text_X_test, url_data=url_X_test)

In [22]:
from sklearn.metrics import classification_report

print(len(text_y_test))
print(len(url_y_test))

print(classification_report(url_y_test, preds))

2000
2000
              precision    recall  f1-score   support

           0       0.44      0.81      0.57       648
           1       0.85      0.51      0.64      1352

    accuracy                           0.61      2000
   macro avg       0.65      0.66      0.60      2000
weighted avg       0.72      0.61      0.62      2000

